# Ollama Notebook

In [1]:
import sys, os, tempfile, time, csv, random
import requests
import pandas as pd
from IPython.display import Image, display

# parsers/ is the cwd when running from here
if "." not in sys.path:
	sys.path.insert(0, ".")

from ollama import OllamaModel, Ollama
from colors import Colors

In [2]:
IMAGES_CSV = "../temp/images.csv"
OUT_DIR    = "../results"
os.makedirs(OUT_DIR, exist_ok=True)

images = pd.read_csv(IMAGES_CSV)
print(f"{len(images)} images loaded")
images.head(3)

2053 images loaded


,Title,Identifier,Artist,Date,Image URL,Alt Text,Is Primary
0,Robert Hamilton Bishop (1777-1855),1829.P.1.1,Horace Harding,1829-1830,https://d1y502jg6fpugt.cloudfront.net/29278/ar...,Robert Hamilton Bishop (1777-1855),1
1,"Portrait of John Williamson Herron, (1827-1912)",1905.P.1.1,"Annette Covington, American 1872-1964",1845,https://d1y502jg6fpugt.cloudfront.net/29278/ar...,"Portrait of John Williamson Herron, (1827-1912)",1
2,Group of Farm Animals with Horse and Rider,1909.P.1.1,James H. Beard,NaN,https://d1y502jg6fpugt.cloudfront.net/29278/ar...,Group of Farm Animals with Horse and Rider,1


In [3]:
import re
from urllib.parse import urlparse

TEMP_DIR = "../temp"

def local_cache_path(url):
	"""Return the path where cache.py would store the full-size image for this URL."""
	parsed = urlparse(url)
	domain = parsed.netloc
	last_segment = [s for s in parsed.path.split("/") if s][-1] if parsed.path else "image"
	basename = re.sub(r"[^a-zA-Z0-9._-]", "_", last_segment)
	return os.path.join(TEMP_DIR, "full", domain, basename + ".jpg")

def download(url):
	"""Return a local path for the image — uses temp cache if already downloaded."""
	cached = local_cache_path(url)
	if os.path.exists(cached):
		return cached, False  # (path, is_temp)

	resp = requests.get(url, timeout=30)
	resp.raise_for_status()
	suffix = ".png" if "png" in url.lower() else ".jpg"
	tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
	tmp.write(resp.content)
	tmp.close()
	return tmp.name, True  # (path, is_temp)

def run(image_url, model=OllamaModel.GEMMA_4, include_colors=True):
	"""Download image (or use cache) and run Ollama + color parser."""
	path, is_temp = download(image_url)
	try:
		result = Ollama().fetch(path, model)
		if include_colors:
			result["colors"] = Colors().fetch(path)
		return result
	finally:
		if is_temp:
			os.unlink(path)

In [4]:
# Quick test — one image
test_url = images["Image URL"].iloc[0]
display(Image(url=test_url))

result = run(test_url)
print("status :", result["status"])
print("model  :", result.get("model"))
print("colors :", len(result.get("colors", [])), "colors extracted")
print()
print(result.get("body", "")[:500])

status : 200
model  : gemma4:latest
colors : 4 colors extracted

Based on the image provided, here is a detailed description:

***

### General Overview

The image is a formal, traditional portrait that appears to be an oil painting or a highly realistic photograph of such a painting. It depicts a single man seated in a serious, thoughtful manner. The style suggests the mid-to-late 19th century, characterized by dark, rich tones and dramatic studio staging.

### The Subject

The central figure is a man who appears to be middle-aged. He possesses a neat hairst


# Benchmark — 100 random images, both models

In [4]:
SAMPLE_SIZE   = 100
SEED          = 1809
BENCHMARK_CSV = f"{OUT_DIR}/benchmark.csv"
MODELS        = [OllamaModel.GEMMA_4, OllamaModel.GEMMA_4_26B]

# Same 100 images for both models
sample = images.sample(SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)
print(f"Sample locked to seed={SEED}: {len(sample)} images")

Sample locked to seed=1809: 100 images


In [6]:
BENCHMARK_CSV = f"{OUT_DIR}/benchmark.csv"
MODELS        = [OllamaModel.GEMMA_4, OllamaModel.GEMMA_4_26B]

rows = []

for model in MODELS:
	print(f"\n### {model.name}  ({model.model_id})")
	for i, row in sample.iterrows():
		image_url = row["Image URL"]
		t0 = time.perf_counter()
		result = run(image_url, model=model, include_colors=False)
		elapsed = time.perf_counter() - t0

		body = result.get("body") or ""
		cold = " (cold start)" if i == 0 else ""
		flag = "" if result["status"] == 200 else "  <-- ERROR"
		print(f"  [{i+1:3}] {row['Identifier']:<20} {elapsed:6.1f}s  {len(body):5d} chars{cold}{flag}")

		rows.append({
			"model":       model.name,
			"identifier":  row["Identifier"],
			"title":       row["Title"],
			"status":      result["status"],
			"runtime_s":   round(elapsed, 1),
			"chars":       len(body),
			"description": body,
		})

	# Save after each model so progress isn't lost if interrupted
	pd.DataFrame(rows).to_csv(BENCHMARK_CSV, index=False)

df_results = pd.DataFrame(rows)
print(f"\nDone — {len(df_results)} rows saved to {BENCHMARK_CSV}")
df_results[["model", "identifier", "status", "runtime_s", "chars"]]


### gemma4  (gemma4:latest)
  [  1] obj-02155              31.6s   1598 chars (cold start)
  [  2] 1993.46                48.4s   2855 chars
  [  3] 2024.11                31.6s   1697 chars
  [  4] 2006.321               39.3s   2201 chars
  [  5] 2023.21                39.7s   1863 chars
  [  6] 1996.47                21.0s   1698 chars
  [  7] 1978.C.2.119           31.4s   1757 chars
  [  8] 1981.156               34.7s   1919 chars
  [  9] 2006.527               38.2s   2051 chars
  [ 10] 2019.23.11             40.6s   2000 chars
  [ 11] 1996.76                35.7s   1680 chars
  [ 12] 2013.HB.99             30.1s   1601 chars
  [ 13] 1996.30                17.3s   1802 chars
  [ 14] 2013.HB.38             36.6s   1976 chars
  [ 15] 2000.101               39.8s   2402 chars
  [ 16] 1978.S.2.48            35.8s   2011 chars
  [ 17] 1996.144               39.7s   2457 chars
  [ 18] 2006.202               44.9s   2355 chars
  [ 19] 2006.175               36.2s   2039 chars
  [ 20] 

,model,identifier,status,runtime_s,chars
0,gemma4,obj-02155,200,31.6,1598
1,gemma4,1993.46,200,48.4,2855
2,gemma4,2024.11,200,31.6,1697
3,gemma4,2006.321,200,39.3,2201
4,gemma4,2023.21,200,39.7,1863
...,...,...,...,...,...
195,gemma4-26b,2010.67,200,39.7,946
196,gemma4-26b,1996.6,200,36.7,1119
197,gemma4-26b,1992.36,200,40.9,1415
198,gemma4-26b,1986.146,200,34.8,1524


In [5]:
df_results = pd.read_csv(BENCHMARK_CSV)
df_results.groupby("model")[["runtime_s", "chars"]].mean().round(1)

,runtime_s,chars
model,,
gemma4,34.7,2038.8
gemma4-26b,37.3,1277.7


In [ ]:
df_results.groupby("model")[["runtime_s"]].sum() / 60

,runtime_s
model,
gemma4,3467.9
gemma4-26b,3725.8
